# Notification Fatigue Optimization: Data Understanding and Validation

**Goal:** Establish the actual source schema, grain, keys, target eligibility, timestamp coverage, and leakage constraints before analysis. The data is synthetic, so findings describe the simulation rather than real-world causal effects.

## Imports

Import only the approved analysis libraries and small standard-library utilities needed for validation.

In [1]:
from pathlib import Path
import hashlib
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
# Self-contained analysis helpers integrated from the former phase modules.

# Only the dependency closure used by this notebook is included.

import json

OBSERVATION_END = pd.Timestamp("2026-04-01")

KEY = ["user_id", "event_timestamp"]

CHURN_METADATA = [
    "source_churn_target", "source_no_app_open_next_30d", "churn_status",
    "churn_witness_event_timestamp", "churn_witness_latest_open",
    "churn_window_complete_calendar", "churn_model_eligible",
]

def validate_churn_labels(raw, observation_end=OBSERVATION_END):
    """Retain only evidence-backed negatives; never infer inactivity from silence."""
    frame = raw.copy().reset_index(drop=True)
    frame["event_timestamp"] = pd.to_datetime(frame["event_timestamp"], errors="raise")
    if frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
        raise ValueError("Churn evidence requires unique, nonmissing event keys")
    if "source_churn_target" in frame:
        raise ValueError("Pass immutable raw events, not an already validated table")
    frame["source_churn_target"] = frame["churn_target"]
    frame["source_no_app_open_next_30d"] = frame["no_app_open_next_30d"]
    witness = pd.Series(pd.NaT, index=frame.index, dtype="datetime64[ns]")
    for _, group in frame.groupby("user_id", sort=False):
        opens = np.sort(group.loc[group["app_opened_24h"].eq(True), "event_timestamp"].to_numpy())
        times = group["event_timestamp"].to_numpy()
        # A later event strictly more than 24h after t avoids overlap ambiguity.
        # Its entire possible open interval must lie inside (t, t + 30d].
        positions = np.searchsorted(opens, times + np.timedelta64(1, "D"), side="right")
        available = positions < len(opens)
        selected = np.full(len(group), np.datetime64("NaT"), dtype="datetime64[ns]")
        selected[available] = opens[positions[available]]
        inside = available & (selected + np.timedelta64(1, "D") <= times + np.timedelta64(30, "D"))
        selected[~inside] = np.datetime64("NaT")
        witness.loc[group.index] = selected
    frame["churn_witness_event_timestamp"] = witness
    frame["churn_witness_latest_open"] = witness + pd.Timedelta(days=1)
    mature = frame["event_timestamp"] + pd.Timedelta(days=30) <= pd.Timestamp(observation_end)
    frame["churn_window_complete_calendar"] = mature
    source_missing = frame[["source_churn_target", "source_no_app_open_next_30d"]].isna().any(axis=1)
    supported = mature & ~source_missing & witness.notna()
    frame["churn_status"] = "unverified_incomplete_activity_history"
    frame.loc[~mature, "churn_status"] = "censored_window"
    frame.loc[source_missing, "churn_status"] = "source_missing_preserved"
    frame.loc[supported, "churn_status"] = "verified_future_open_negative"
    corrected = pd.Series(pd.NA, index=frame.index, dtype="Float64")
    corrected.loc[supported] = 0.0
    frame["churn_target"] = corrected
    frame["no_app_open_next_30d"] = corrected.copy()
    # Observing a subset of negatives does not create a representative risk set.
    frame["churn_model_eligible"] = False
    assert_validated_churn(frame)
    return frame

def assert_validated_churn(frame):
    """Fail closed on unverified targets, positive claims, or missing witnesses."""
    required = CHURN_METADATA + ["churn_target", "no_app_open_next_30d"]
    if not set(required).issubset(frame.columns):
        raise ValueError("Unvalidated churn labels: use load_events()")
    observed = frame["churn_target"].notna()
    if frame.loc[observed, "churn_target"].ne(0).any():
        raise ValueError("Positive churn cannot be established from incomplete activity")
    if frame["churn_model_eligible"].any():
        raise ValueError("No complete activity-history population is available")
    if not frame["churn_target"].equals(frame["no_app_open_next_30d"]):
        raise ValueError("Canonical churn columns disagree")
    if not frame.loc[observed, "churn_status"].eq("verified_future_open_negative").all():
        raise ValueError("Unverified label cannot be retained")
    t = pd.to_datetime(frame.loc[observed, "event_timestamp"])
    w = pd.to_datetime(frame.loc[observed, "churn_witness_event_timestamp"])
    if not ((w > t + pd.Timedelta(days=1)) & (w + pd.Timedelta(days=1) <= t + pd.Timedelta(days=30))).all():
        raise ValueError("Churn witness outside conservative future-activity window")
    if frame.loc[frame["source_churn_target"].isna(), "churn_target"].notna().any():
        raise ValueError("Source missingness must be preserved")

def churn_training_data(frame):
    """Block population modeling until complete independent activity is supplied."""
    assert_validated_churn(frame)
    raise ValueError("Churn modeling blocked: incomplete activity history and evidence-selected negatives")

def load_events(project_dir):
    """Load original events through the mandatory churn-validation boundary."""
    return validate_churn_labels(pd.read_csv(Path(project_dir) / "data/raw/notification_events.csv"))

def build_churn_artifacts(project_dir):
    """Update only churn fields/provenance in the existing processed event table."""
    root = Path(project_dir)
    events = load_events(root)
    columns = KEY + ["churn_target", "no_app_open_next_30d"] + CHURN_METADATA
    labels = events[columns]
    labels.to_csv(root / "data/processed/churn_label_validation.csv", index=False)
    feature_path = root / "data/processed/notification_features.csv"
    # Preserve all unrelated serialized feature values, including their precision.
    features = pd.read_csv(feature_path, dtype=str, keep_default_na=False)
    if len(features) != len(events) or not features["user_id"].eq(events["user_id"].astype(str)).all():
        raise ValueError("Processed event IDs do not align with raw events")
    if not pd.to_datetime(features["event_timestamp"]).eq(events["event_timestamp"]).all():
        raise ValueError("Processed timestamps do not align with raw events")
    for name in columns[2:]:
        features[name] = labels[name].astype("string").fillna("")
    features.to_csv(feature_path, index=False)
    summary = {
        "raw_sha256": hashlib.sha256((root / "data/raw/notification_events.csv").read_bytes()).hexdigest(),
        "observation_end_exclusive": str(OBSERVATION_END),
        "statuses": events["churn_status"].value_counts().to_dict(),
        "corrected_negative_labels": int(events["churn_target"].eq(0).sum()),
        "model_eligible_rows": int(events["churn_model_eligible"].sum()),
        "complete_activity_history_available": False,
    }
    (root / "data/processed/churn_validation_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return events, summary

def validated_timestamps(frame):
    """Reject malformed, absent, or timezone-mixed source clocks."""
    stamps = pd.to_datetime(frame["event_timestamp"], errors="raise")
    if stamps.isna().any() or not isinstance(stamps.dtype, np.dtype) or stamps.dtype.kind != "M":
        raise ValueError("Event timestamps must be complete local naive datetimes")
    if "event_date" in frame and not pd.to_datetime(frame["event_date"], errors="raise").eq(stamps.dt.normalize()).all():
        raise ValueError("event_date disagrees with event_timestamp")
    if "hour_local" in frame and not frame["hour_local"].eq(stamps.dt.hour).all():
        raise ValueError("hour_local disagrees with event_timestamp")
    return stamps

def exposure_counts(events):
    """Delivered notifications in [local event date - D calendar days, event date)."""
    stamps = validated_timestamps(events)
    if events[["user_id", "event_timestamp"]].isna().any().any() or events.duplicated(["user_id", "event_timestamp"]).any():
        raise ValueError("Exposure requires unique event keys")
    dates = stamps.dt.normalize()
    out = {label: np.zeros(len(events), dtype=np.int64) for label in ["24h", "7d", "30d"]}
    date_values = dates.to_numpy(dtype="datetime64[ns]")
    delivery_values = events["delivered"].to_numpy()
    for positions in events.groupby("user_id", sort=False).indices.values():
        positions = np.asarray(positions)
        delivered = np.sort(date_values[positions[delivery_values[positions]]])
        current = date_values[positions]
        upper = np.searchsorted(delivered, current, side="left")
        for days, label in [(1, "24h"), (7, "7d"), (30, "30d")]:
            out[label][positions] = upper - np.searchsorted(delivered, current - np.timedelta64(days, "D"), side="left")
    return pd.DataFrame({f"notifications_prior_{label}": values for label, values in out.items()}, index=events.index)


The environment is configured for compact, reproducible inspection.

## Configuration and provenance

Use the exact CSVs currently stored in `data/raw/`. A cryptographic digest identifies the files used without changing them.

In [2]:
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

user_path = project_dir / "data" / "raw" / "users.csv"
event_path = project_dir / "data" / "raw" / "notification_events.csv"

for path in [user_path, event_path]:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path.name}: {path.stat().st_size:,} bytes | SHA256 {digest}")

users.csv: 230,449 bytes | SHA256 0645e7dadfa418d4c9d18260e8b47e382eb21e285052fc6a04558372e7da53de
notification_events.csv: 14,160,146 bytes | SHA256 d5e57865466eb33286af0f7444d174e0667f21bddccc803715daed5edaf39eba


The printed filenames and digests are the reproducible provenance for this run; the original pre-repository filenames cannot be verified from the available files.

## Data Cleaning and Preprocessing

Load the raw tables without coercing missing outcomes or modifying source rows.

In [3]:
users = pd.read_csv(user_path)
events, churn_summary = build_churn_artifacts(project_dir)

events["event_timestamp"] = pd.to_datetime(events["event_timestamp"], errors="coerce")
events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")

Raw files remain immutable. The in-memory and processed views quarantine unverifiable churn labels, preserve source values, and retain only evidence-backed negative labels. Counts now include validation metadata.

### First rows

Inspect representative records before calculating any metrics.

In [4]:
display(users.head())
display(events.head())

,user_id,user_segment,preferred_channel,tenure_days_at_start,baseline_activity_probability,engagement_propensity,fatigue_sensitivity,notification_opt_in_at_start
0,1,low_activity,email,872,0.10,0.4185,0.4699,True
1,2,low_activity,in_app,560,0.10,0.3211,0.4095,True
2,3,casual,in_app,57,0.27,0.1051,0.2060,True
3,4,regular,push,351,0.52,0.7594,0.3689,True
4,5,power,email,160,0.75,0.5628,0.7900,True


,user_id,event_timestamp,channel,notification_type,content_relevance,personalized,notifications_prior_24h,notifications_prior_7d,notifications_prior_30d,fatigue_before_send,inactive_days_streak,app_active_on_event_day,delivered,suppression_reason,clicked_24h,app_opened_24h,engaged_24h,unsubscribed_at_event,converted_7d,no_app_open_next_30d,hour_local,event_date,fatigue_after_event,engagement_target,churn_target,source_churn_target,source_no_app_open_next_30d,churn_witness_event_timestamp,churn_witness_latest_open,churn_window_complete_calendar,churn_status,churn_model_eligible
0,7,2026-01-01 18:00:00,push,reminder,0.566988,False,0,0,0,0.129639,1,False,True,NaN,False,False,False,False,False,0.0,18,2026-01-01,0.2296,0,0.0,1.0,1.0,2026-01-11 17:00:00,2026-01-12 17:00:00,True,verified_future_open_negative,False
1,12,2026-01-01 11:00:00,email,social,0.441400,False,0,0,0,0.048334,0,True,True,NaN,True,True,True,False,True,0.0,11,2026-01-01,0.1033,1,0.0,1.0,1.0,2026-01-10 14:00:00,2026-01-11 14:00:00,True,verified_future_open_negative,False
2,26,2026-01-01 16:00:00,push,recommendation,0.582710,True,0,0,0,0.075336,1,False,True,NaN,False,False,False,False,False,0.0,16,2026-01-01,0.1753,0,0.0,1.0,1.0,2026-01-03 16:00:00,2026-01-04 16:00:00,True,verified_future_open_negative,False
3,27,2026-01-01 19:00:00,push,promotion,0.630184,True,0,0,0,0.192297,1,False,True,NaN,False,False,False,False,False,0.0,19,2026-01-01,0.2923,0,0.0,1.0,1.0,2026-01-30 11:00:00,2026-01-31 11:00:00,True,verified_future_open_negative,False
4,31,2026-01-01 14:00:00,push,social,0.565946,True,0,0,0,0.048027,0,True,True,NaN,True,True,True,False,False,<NA>,14,2026-01-01,0.1030,1,<NA>,1.0,1.0,NaT,NaT,True,unverified_incomplete_activity_history,False


The user table is profile-level, while the event table contains candidate notification decisions and downstream outcomes.

### Shape

Print rows and columns separately to verify the actual files against the documentation.

In [5]:
print(f"Users rows: {users.shape[0]:,}")
print(f"Users columns: {users.shape[1]:,}")
print(f"Event rows: {events.shape[0]:,}")
print(f"Event columns: {events.shape[1]:,}")

Users rows: 5,000
Users columns: 8
Event rows: 85,019
Event columns: 32


The executed shapes replace the earlier approximate schema descriptions for all downstream work.

### Data types

Check parsed types before using identifiers, booleans, numbers, or timestamps.

In [6]:
user_info = io.StringIO()
event_info = io.StringIO()
users.info(buf=user_info)
events.info(buf=event_info)
print("USERS INFO")
print(user_info.getvalue())
print("EVENTS INFO")
print(event_info.getvalue())

USERS INFO
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   user_id                        5000 non-null   int64  
 1   user_segment                   5000 non-null   str    
 2   preferred_channel              5000 non-null   str    
 3   tenure_days_at_start           5000 non-null   int64  
 4   baseline_activity_probability  5000 non-null   float64
 5   engagement_propensity          5000 non-null   float64
 6   fatigue_sensitivity            5000 non-null   float64
 7   notification_opt_in_at_start   5000 non-null   bool   
dtypes: bool(1), float64(3), int64(2), str(2)
memory usage: 336.1 KB

EVENTS INFO
<class 'pandas.DataFrame'>
RangeIndex: 85019 entries, 0 to 85018
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          ---------

Identifiers and numeric features load as numeric values; the two event date fields are explicitly parsed as datetimes.

### Descriptive summaries

Review ranges and categories before defining bins, filters, or targets.

In [7]:
display(users.describe(include="all").T)
display(events.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,5000.0,NaN,NaN,NaN,2500.5,1443.520003,1.0,1250.75,2500.5,3750.25,5000.0
user_segment,5000,4,casual,1885,NaN,NaN,NaN,NaN,NaN,NaN,NaN
preferred_channel,5000,3,push,3113,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure_days_at_start,5000.0,NaN,NaN,NaN,450.9346,257.682435,7.0,224.75,454.0,676.0,899.0
baseline_activity_probability,5000.0,NaN,NaN,NaN,0.377012,0.198675,0.1,0.27,0.27,0.52,0.75
engagement_propensity,5000.0,NaN,NaN,NaN,0.450331,0.209421,0.03,0.290575,0.4413,0.600425,0.95
fatigue_sensitivity,5000.0,NaN,NaN,NaN,0.469192,0.217287,0.0055,0.3042,0.4645,0.63625,0.9757
notification_opt_in_at_start,5000,1,True,5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
user_id,85019.0,NaN,NaN,NaN,2499.11201,1.0,1258.0,2495.0,3741.5,5000.0,1441.989437
event_timestamp,85019,NaN,NaN,NaN,2026-02-14 12:18:19.575389,2026-01-01 07:00:00,2026-01-22 20:00:00,2026-02-13 22:00:00,2026-03-08 22:00:00,2026-03-31 22:00:00,NaN
channel,85019,3,push,52948,NaN,NaN,NaN,NaN,NaN,NaN,NaN
notification_type,85019,4,recommendation,28951,NaN,NaN,NaN,NaN,NaN,NaN,NaN
content_relevance,85019.0,NaN,NaN,NaN,0.600524,0.012476,0.457751,0.615164,0.756557,0.999255,0.199579
personalized,85019,2,True,47679,NaN,NaN,NaN,NaN,NaN,NaN,NaN
notifications_prior_24h,85019.0,NaN,NaN,NaN,0.144873,0.0,0.0,0.0,0.0,1.0,0.351975
notifications_prior_7d,85019.0,NaN,NaN,NaN,0.967737,0.0,0.0,1.0,1.0,4.0,0.822223
notifications_prior_30d,85019.0,NaN,NaN,NaN,3.582435,0.0,2.0,4.0,5.0,11.0,2.084443
fatigue_before_send,85019.0,NaN,NaN,NaN,0.125996,0.000127,0.078632,0.120873,0.167011,0.471758,0.063018


The summaries expose the observed ranges and category cardinalities without assuming values from the earlier data dictionary.

### Missing values

Missingness can encode suppression or right-censoring, so inspect it before deciding whether any row should be removed.

In [8]:
user_missing = users.isna().sum().rename("missing_rows")
event_missing = events.isna().sum().rename("missing_rows")
print("USER MISSINGNESS")
display(user_missing[user_missing.gt(0)].to_frame())
print("EVENT MISSINGNESS")
display(event_missing[event_missing.gt(0)].to_frame())

USER MISSINGNESS


,missing_rows


EVENT MISSINGNESS


,missing_rows
suppression_reason,59341
no_app_open_next_30d,41997
churn_target,41997
source_churn_target,27591
source_no_app_open_next_30d,27591
churn_witness_event_timestamp,29229
churn_witness_latest_open,29229


Missing suppression reasons correspond to delivered rows. Missing corrected churn values represent either source censoring/missingness or unverified source labels; churn_status distinguishes them. No blanket dropna() is appropriate.

### Duplicate rows and keys

Check full-row duplicates and the available business keys. The event file has no event identifier, so the timestamp key must be tested directly.

In [9]:
print(f"Duplicate user rows: {users.duplicated().sum():,}")
print(f"Duplicate event rows: {events.duplicated().sum():,}")
print(f"Duplicate user IDs: {users['user_id'].duplicated().sum():,}")
print(
    "Duplicate event user-timestamps: "
    f"{events.duplicated(['user_id', 'event_timestamp']).sum():,}"
)

Duplicate user rows: 0
Duplicate event rows: 0
Duplicate user IDs: 0
Duplicate event user-timestamps: 0


The available keys are unique, so no deduplication is applied.

## Column Description

List each observed column with its executed dtype and a concise role derived from its name and values.

In [10]:
user_roles = {
    "user_id": "user key",
    "user_segment": "baseline segment",
    "preferred_channel": "baseline preferred channel",
    "tenure_days_at_start": "tenure before observation",
    "baseline_activity_probability": "baseline activity propensity",
    "engagement_propensity": "baseline engagement propensity",
    "fatigue_sensitivity": "baseline fatigue sensitivity",
    "notification_opt_in_at_start": "baseline opt-in state",
}

event_roles = {
    "user_id": "user key",
    "event_timestamp": "candidate notification time",
    "channel": "candidate channel",
    "notification_type": "candidate content type",
    "content_relevance": "candidate relevance score",
    "personalized": "candidate personalization flag",
    "notifications_prior_24h": "labeled prior exposure count",
    "notifications_prior_7d": "labeled prior exposure count",
    "notifications_prior_30d": "labeled prior exposure count",
    "fatigue_before_send": "labeled pre-decision fatigue score",
    "inactive_days_streak": "labeled pre-event inactivity streak",
    "app_active_on_event_day": "same-day activity flag; timing unresolved",
    "delivered": "observed delivery decision",
    "suppression_reason": "observed suppression reason",
    "clicked_24h": "post-event click outcome",
    "app_opened_24h": "post-event app-open outcome",
    "engaged_24h": "post-event engagement outcome",
    "unsubscribed_at_event": "post-event unsubscribe outcome",
    "converted_7d": "post-event conversion outcome",
    "no_app_open_next_30d": "future inactivity label",
    "hour_local": "candidate local hour",
    "event_date": "candidate calendar date",
    "fatigue_after_event": "post-event fatigue score",
    "engagement_target": "numeric copy of engagement outcome",
    "churn_target": "future churn label",
}

for name, frame, roles in [("USERS", users, user_roles), ("EVENTS", events, event_roles)]:
    print(name)
    for column in frame.columns:
        print(f"- {column} ({frame[column].dtype}): {roles.get(column, 'F01 validation provenance; never a predictive feature')}")

USERS
- user_id (int64): user key
- user_segment (str): baseline segment
- preferred_channel (str): baseline preferred channel
- tenure_days_at_start (int64): tenure before observation
- baseline_activity_probability (float64): baseline activity propensity
- engagement_propensity (float64): baseline engagement propensity
- fatigue_sensitivity (float64): baseline fatigue sensitivity
- notification_opt_in_at_start (bool): baseline opt-in state
EVENTS
- user_id (int64): user key
- event_timestamp (datetime64[us]): candidate notification time
- channel (str): candidate channel
- notification_type (str): candidate content type
- content_relevance (float64): candidate relevance score
- personalized (bool): candidate personalization flag
- notifications_prior_24h (int64): labeled prior exposure count
- notifications_prior_7d (int64): labeled prior exposure count
- notifications_prior_30d (int64): labeled prior exposure count
- fatigue_before_send (float64): labeled pre-decision fatigue score


This executed inventory is the schema used by later notebooks; timing-ambiguous fields remain marked rather than silently accepted.

## EDA: Integrity checks

Validation here focuses on joins, timestamps, decision status, targets, and pre-event feature evidence.

### User-to-event join coverage

Confirm all event users map to the profile table and identify profile rows without events.

In [11]:
event_users = set(events["user_id"])
profile_users = set(users["user_id"])
print(f"Event users found in profiles: {len(event_users & profile_users):,} of {len(event_users):,}")
print(f"Profile users without events: {len(profile_users - event_users):,}")

Event users found in profiles: 4,995 of 4,995
Profile users without events: 5


All event rows are joinable. Profiles without events remain valid users and are retained for population context.

### Timestamp coverage and ordering

Verify the observation window, parse failures, global ordering, and within-user timestamp uniqueness.

In [12]:
print(f"Minimum event timestamp: {events['event_timestamp'].min()}")
print(f"Maximum event timestamp: {events['event_timestamp'].max()}")
print(f"Unparsed event timestamps: {events['event_timestamp'].isna().sum():,}")
print(f"Globally sorted in source order: {events['event_timestamp'].is_monotonic_increasing}")
print(f"Duplicate within-user timestamps: {events.duplicated(['user_id', 'event_timestamp']).sum():,}")

Minimum event timestamp: 2026-01-01 07:00:00
Maximum event timestamp: 2026-03-31 22:00:00
Unparsed event timestamps: 0
Globally sorted in source order: False
Duplicate within-user timestamps: 0


The file covers the documented calendar window but is not globally sorted, so all longitudinal calculations must sort by user and timestamp first.

### Categories and decision status

Enumerate categories and reconcile delivery with the suppression reason.

In [13]:
for column in ["user_segment", "preferred_channel"]:
    print(f"{column}: {users[column].value_counts(dropna=False).to_dict()}")

for column in ["channel", "notification_type"]:
    print(f"{column}: {events[column].value_counts(dropna=False).to_dict()}")

display(pd.crosstab(
    events["delivered"],
    events["suppression_reason"].fillna("missing"),
    margins=True,
))

user_segment: {'casual': 1885, 'regular': 1713, 'low_activity': 871, 'power': 531}
preferred_channel: {'push': 3113, 'email': 1161, 'in_app': 726}
channel: {'push': 52948, 'email': 19892, 'in_app': 12179}
notification_type: {'recommendation': 28951, 'reminder': 22849, 'promotion': 21093, 'social': 12126}


suppression_reason,daily_cap,missing,quiet_hours,All
delivered,,,,
False,12317,0,13361,25678
True,0,59341,0,59341
All,12317,59341,13361,85019


Delivery status reconciles exactly with missing versus populated suppression reasons, supporting `delivered` as the eligibility flag for response-rate denominators.

### Response-target values and denominators

Inspect binary outcomes and confirm whether suppressed rows can contribute responses.

In [14]:
target_columns = [
    "clicked_24h", "app_opened_24h", "engaged_24h",
    "unsubscribed_at_event", "converted_7d", "engagement_target",
]
for column in target_columns:
    print(f"{column}: {events[column].value_counts(dropna=False).to_dict()}")

display(events.groupby("delivered")[target_columns[:-1]].sum())
print(
    "engagement_target equals engaged_24h: "
    f"{(events['engagement_target'] == events['engaged_24h'].astype(int)).mean():.2%}"
)

clicked_24h: {False: 65852, True: 19167}
app_opened_24h: {False: 64516, True: 20503}
engaged_24h: {False: 63539, True: 21480}
unsubscribed_at_event: {False: 84908, True: 111}
converted_7d: {False: 82952, True: 2067}
engagement_target: {0: 63539, 1: 21480}


,clicked_24h,app_opened_24h,engaged_24h,unsubscribed_at_event,converted_7d
delivered,,,,,
False,0,0,0,0,0
True,19167,20503,21480,111,2067


engagement_target equals engaged_24h: 100.00%


Clicks and engagement occur only on delivered events, so later CTR and engagement rates use delivered events as their denominator. The numeric engagement target duplicates the boolean outcome.

### Churn label investigation

The original positive labels contradict later app opens. A conservative witness must start more than 24 hours after the candidate, and its full 24-hour open window must finish within 30 days. Source-missing and immature labels remain missing; absence of a witness never establishes churn.

In [15]:
print("Original churn values (provenance only)")
print(events["source_churn_target"].value_counts(dropna=False))
print("Corrected label handling")
display(events["churn_status"].value_counts().to_frame("rows"))
print("Evidence-backed negative labels:", events["churn_target"].eq(0).sum())
print("Positive labels retained:", events["churn_target"].eq(1).sum())
print("Model-eligible rows:", events["churn_model_eligible"].sum())
display(events.loc[events["churn_target"].notna(), [
    "user_id", "event_timestamp", "source_churn_target", "churn_target",
    "churn_witness_event_timestamp", "churn_witness_latest_open",
]].head())
print(churn_summary)

Original churn values (provenance only)
source_churn_target
1.0    57428
NaN    27591
Name: count, dtype: int64
Corrected label handling


,rows
churn_status,
verified_future_open_negative,43022
source_missing_preserved,27591
unverified_incomplete_activity_history,14406


Evidence-backed negative labels: 43022
Positive labels retained: 0
Model-eligible rows: 0


,user_id,event_timestamp,source_churn_target,churn_target,churn_witness_event_timestamp,churn_witness_latest_open
0,7,2026-01-01 18:00:00,1.0,0.0,2026-01-11 17:00:00,2026-01-12 17:00:00
1,12,2026-01-01 11:00:00,1.0,0.0,2026-01-10 14:00:00,2026-01-11 14:00:00
2,26,2026-01-01 16:00:00,1.0,0.0,2026-01-03 16:00:00,2026-01-04 16:00:00
3,27,2026-01-01 19:00:00,1.0,0.0,2026-01-30 11:00:00,2026-01-31 11:00:00
6,75,2026-01-01 07:00:00,1.0,0.0,2026-01-14 13:00:00,2026-01-15 13:00:00


{'raw_sha256': 'd5e57865466eb33286af0f7444d174e0667f21bddccc803715daed5edaf39eba', 'observation_end_exclusive': '2026-04-01 00:00:00', 'statuses': {'verified_future_open_negative': 43022, 'source_missing_preserved': 27591, 'unverified_incomplete_activity_history': 14406}, 'corrected_negative_labels': 43022, 'model_eligible_rows': 0, 'complete_activity_history_available': False}


F01 is contained, but full reconstruction remains unavailable: no complete activity log or generator is supplied. Supported negatives are selected by observed activity and cannot form a representative churn-model population. All positive or otherwise unverified source labels are quarantined.

### Historical feature timing audit

Reconstruct the three exposure fields from delivered notifications in prior complete local calendar days. Then inspect fatigue separately; its generator remains unavailable.

In [16]:
reconstructed_exposure = exposure_counts(events)
for column in reconstructed_exposure:
    mismatches = int(events[column].ne(reconstructed_exposure[column]).sum())
    print(f"{column}: {mismatches:,} mismatches across {len(events):,} candidate events")
    assert mismatches == 0

sorted_events = events.sort_values(["user_id", "event_timestamp"]).copy()
previous_fatigue = sorted_events.groupby("user_id")["fatigue_after_event"].shift()
has_previous = previous_fatigue.notna()

print("Delivered-event prior-count ranges")
display(
    sorted_events.loc[sorted_events["delivered"], [
        "notifications_prior_24h", "notifications_prior_7d", "notifications_prior_30d"
    ]].describe().T
)
print(
    "Correlation of fatigue_before_send with the previous row's fatigue_after_event: "
    f"{sorted_events.loc[has_previous, 'fatigue_before_send'].corr(previous_fatigue[has_previous]):.3f}"
)
print(
    "Exact agreement within rounding tolerance: "
    f"{np.isclose(sorted_events.loc[has_previous, 'fatigue_before_send'], previous_fatigue[has_previous], atol=1e-4).mean():.2%}"
)

notifications_prior_24h: 0 mismatches across 85,019 candidate events
notifications_prior_7d: 0 mismatches across 85,019 candidate events
notifications_prior_30d: 0 mismatches across 85,019 candidate events
Delivered-event prior-count ranges


,count,mean,std,min,25%,50%,75%,max
notifications_prior_24h,59341.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
notifications_prior_7d,59341.0,0.837667,0.766496,0.0,0.0,1.0,1.0,3.0
notifications_prior_30d,59341.0,3.430377,2.052314,0.0,2.0,3.0,5.0,11.0


Correlation of fatigue_before_send with the previous row's fatigue_after_event: 0.776
Exact agreement within rounding tolerance: 0.03%


All three exposure counts reconstruct exactly as delivered notifications in `[event_date - D calendar days, event_date)`, where D is 1, 7, or 30. The 24h suffix is historical source naming, not a rolling elapsed 24-hour window. Timestamps use the supplied local clock with no verified timezone. The same-day candidate is excluded. Fatigue construction and its availability time remain unverified.

## Problem Statement

Use delivered-event responses for engagement analysis. Churn modeling and population churn conclusions remain blocked by incomplete activity history; a verified negative does not make the remaining unknown labels positive. Policy outcomes remain explicitly simulated.

## Summary

Print the validated facts that control all downstream denominators and eligibility decisions.

In [17]:
print(f"Validated profiles: {len(users):,}")
print(f"Validated candidate events: {len(events):,}")
print(f"Delivered response opportunities: {events['delivered'].sum():,}")
print(f"Suppressed opportunities: {(~events['delivered']).sum():,}")
print(f"Observed churn classes: {events.loc[events['churn_target'].notna(), 'churn_target'].nunique():,}")
print(f"Event-user join coverage: {events['user_id'].isin(users['user_id']).mean():.2%}")

Validated profiles: 5,000
Validated candidate events: 85,019
Delivered response opportunities: 59,341
Suppressed opportunities: 25,678
Observed churn classes: 1
Event-user join coverage: 100.00%


Join and delivery checks remain valid. The corrected churn view contains supported negatives and missing outcomes, with provenance and an explicit modeling block. Raw source labels are invalid for business use.

## Insights

- Use delivered events for notification response rates.
- Use only the validated churn view; never use the archived source labels for modeling or churn-rate claims.
- Complete activity history is required to recover a representative 30-day churn outcome.
- F02 state transitions are implemented in Notebook 05; all results remain synthetic scenarios.